This notebook shows how to use the clemcore Gymnasium integration by the example of the multi-player game Taboo.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](colab.research.google.com)

In [6]:
import os

# Specify where to locate the clembench repo
CLEMBENCH_HOME = os.path.expanduser("~/git/clembench")

# Expose the CLEMBENCH_HOME variable (this directory will be looked up for the games)
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

In [7]:
# If not done yet, clone the clembench repo to CLEMBENCH_HOME
!git clone https://github.com/clp-research/clembench $CLEMBENCH_HOME

# Install the requirements into the Python kernel
%pip install -r $CLEMBENCH_HOME/requirements.txt

# Make tqdm usable in jupyter notebooks
%pip install --upgrade ipywidgets jupyter_client

fatal: destination path '/Users/philippsadler/git/clembench' already exists and is not an empty directory.

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
# Sanity check, list games visible via the CLI (through the CLEMBENCH_HOME environment variable)
import clemcore.cli as cli

print("clemcore:", cli.get_version())
cli.list_games("taboo", verbose=False)  # to see all games pass "all"

clemcore: 3.3.5
Listing all available games (use -v option to see the whole specs)
Found '1' game specs that match the game_selector='{'game_name': 'taboo'}'
taboo:
 	Taboo game between two agents where one has to describe a word for
	the other to guess.


In [9]:
# Let's get started: We load the wordle game as a PettingZoo env
from clemcore.clemgame import gym_env
from clemcore import backends

# Note: single_pass=False means that we can run through all game instances multiple times
# Note: game_instance_filter allows using only specific instances of a game
game_env = gym_env(
    "taboo",
    learner_agent="player_0",
    # this needs a model_registry.json and key.json next to the notebook!
    other_agents={"player_1": backends.load_model("clp-chat")}
)

2025-12-18 17:14:22,162 - clemcore.cli - INFO - Found '1' game matching the game_selector="taboo"
2025-12-18 17:14:22,163 - clemcore.cli - INFO - {
  "game_name": "taboo",
  "description": "Taboo game between two agents where one has to describe a word for the other to guess.",
  "main_game": "taboo",
  "players": 2,
  "image": "none",
  "languages": [
    "en"
  ],
  "benchmark": [
    "0.9",
    "1.0",
    "1.5",
    "2.0"
  ],
  "regression": "small",
  "roles": [
    "Describer",
    "Guesser"
  ],
  "game_path": "/Users/philippsadler/git/clembench/taboo"
}
2025-12-18 17:14:22,164 - clemcore.run - INFO - Loading game benchmark for taboo
2025-12-18 17:14:22,397 - clemcore.run - INFO - Loading game benchmark for taboo took: 0:00:00.231041
2025-12-18 17:14:22,400 - clemcore.run - INFO - Sub-select for taboo experiment high_en instances with game_ids: [1, 5, 10]
2025-12-18 17:14:22,401 - clemcore.run - INFO - Sub-select for taboo experiment medium_en instances with game_ids: [1, 5, 10]

In [12]:
# We define our describer agent
from playpen.agents import ClemAgent, ClemObservation


class MyAgenticDescriber(ClemAgent):

    def act(self, observation: ClemObservation) -> str:
        _response = "CLUE: I cannot tell you exactly what this is about."
        if "apples" in observation.content:
            _response += " But it is NOT about apples."
        if "bananas" in observation.content:
            _response += " But bananas ARE a good guess."
        return _response


describer = MyAgenticDescriber()


.--------------..--------------..--------------..--------------..--------------..--------------..--------------.
|   ______     ||   _____      ||      __      ||  ____  ____  ||   ______     ||  _________   || ____  _____  |
|  |_   __ \   ||  |_   _|     ||     /  \     || |_  _||_  _| ||  |_   __ \   || |_   ___  |  |||_   \|_   _| |
|    | |__) |  ||    | |       ||    / /\ \    ||   \ \  / /   ||    | |__) |  ||   | |_  \_|  ||  |   \ | |   |
|    |  ___/   ||    | |   _   ||   / ____ \   ||    \ \/ /    ||    |  ___/   ||   |  _|  _   ||  | |\ \| |   |
|   _| |_      ||   _| |__/ |  || _/ /    \ \_ ||    _|  |_    ||   _| |_      ||  _| |___/ |  || _| |_\   |_  |
|  |_____|     ||  |________|  |||____|  |____|||   |______|   ||  |_____|     || |_________|  |||_____|\____| |
'--------------''--------------''--------------''--------------''--------------''--------------''--------------'



In [14]:
# Now can do everything automated
describer = MyAgenticDescriber()
game_env = gym_env(
    "taboo",
    learner_agent="player_0",
    other_agents={"player_1": backends.load_model("gpt4o")}
)
obs, info = game_env.reset()
termination = False
context_response_pairs: list[tuple] = []
while not termination:
    action = describer(obs)
    context_response_pairs.append((obs, action))
    obs, reward, termination, truncation, info = game_env.step(action)
describer.reset()

print(f"Episode took these {len(context_response_pairs)} steps:")
print("-" * 20)
for idx, (context, response) in enumerate(context_response_pairs):
    print(f"Step {idx}:")
    print(f"Describer <- Context:", context)
    print(f"Describer -> Response:", response)
    print("-" * 20)


Episode took these 6 steps:
--------------------
Step 0:
Agent(player_0) <- Context: {'role': 'user', 'content': 'You are playing a collaborative word guessing game in which you have to describe a target word for another player to guess.\n\nRules:\n(a) You have to reply in the form: CLUE: <some text>. Guesses from the other player will start with GUESS.\n(b) You cannot use the target word itself, parts or morphological variants of it in your description.\n(c) In addition, the same rules apply for related words which are provided below.\n\nEnd conditions:\n(i) If you use the target word or a related word in your description, then you lose.\n(ii) If the other player can guess the target word in 3 tries, you both win.\n\nLet us start.\n\nThis is the target word that you need to describe and that the other player needs to guess:\n\nenergy\n\nRelated words are:\n\n- solar\n- power\n- electricity\n\nImportant: You are under time pressure, give short descriptions that are to the point!'}
Agen